# NB1b-UBX — Tile Feature Extraction from H5 (UBELIX)

**Env:** `ubelix`  
**Inputs:**
- `data/processed/cell_stacks.h5` — registered stacks `(10821, 4, 558, 1108)` uint8
- `data/processed/cell_masks.h5` — canonical masks `(10821, 558, 1108)` uint8
- `data/processed/lucia_geometry.parquet` — `npy_index` column maps `cell_name` → H5 row

**Output:** `data/processed/lucia_tile_features.parquet` — 37-col schema (5 stats × 7 channels)

**Note:** Registration is complete and stored in the H5. PNG source images are **NOT** on  
UBELIX and are **NOT** read by this notebook.

**Before running:** set `LUCIA_ROOT` in your Slurm script:
```bash
export LUCIA_ROOT=/storage/homefs/db98d082/ondemand/LUCIA
```

Run cells in order. The sanity gate (Cell 3) must **PASS** before the batch (Cell 5).

In [1]:
# ── Config + imports + tile grid + _tile_features ─────────────────────────────
import os, sys, shutil, importlib, time
import numpy as np
import pandas as pd
import h5py
import joblib
import scipy.stats
from skimage.measure import shannon_entropy
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')

_LUCIA_ROOT = os.environ.get('LUCIA_ROOT',
    '/storage/homefs/db98d082/ondemand/LUCIA')
sys.path.insert(0, os.path.join(_LUCIA_ROOT, 'notebooks_scripts'))

import lucia_common as lc
importlib.reload(lc)
import lucia_registration_v4 as reg
importlib.reload(reg)

# Auto-detect allocated cores (Slurm sets CPU affinity)
try:
    N_JOBS = len(os.sched_getaffinity(0))
except AttributeError:
    N_JOBS = int(os.environ.get('SLURM_CPUS_PER_TASK', 8))
print(f"N_JOBS = {N_JOBS}")

# Paths (staging paths set in Cell 2)
H5_STACKS_SRC    = lc.STACKS_H5
H5_MASKS_SRC     = lc.MASKS_H5
GEOMETRY_PARQUET = lc.GEOMETRY_PARQUET
TILE_PARQUET     = lc.TILE_PARQUET

# Tile grid — overlap param ignored by reg v4; geometric ~2px overlap from cell geometry
FEATURE_TILES = reg.edge_anchored_tile_grid(tile=64, overlap=0.0)
ALL_CHANNELS  = lc.RAW_CHANNELS + lc.SYNTH_CHANNELS  # 7 channels

print(f"LUCIA_ROOT  : {_LUCIA_ROOT}")
print(f"H5_STACKS   : {H5_STACKS_SRC}  exists={os.path.exists(H5_STACKS_SRC)}")
print(f"H5_MASKS    : {H5_MASKS_SRC}  exists={os.path.exists(H5_MASKS_SRC)}")
print(f"Tile count  : {len(FEATURE_TILES)}  (expect 162)")
print(f"Channels    : {ALL_CHANNELS}")
print(f"Schema      : 2 fixed + 5 stats x {len(ALL_CHANNELS)} channels = {2 + 5*len(ALL_CHANNELS)} cols")


def _tile_features(tile_arrays, channel_names):
    """Per-tile statistics for every channel: mean, std, uni, entropy, skew.

    uni_X = std / (|mean| + 1e-6) — coefficient of variation; abs() keeps it
    safe for signed channels (log_el_pl, rs_map, grad_el_hi).
    Schema: 5 stats x 7 channels = 35 cols + is_border + active_frac = 37 total.
    """
    feats = {}
    for ch in channel_names:
        px  = tile_arrays[ch].ravel()
        m   = float(px.mean())
        s   = float(px.std())
        feats[f"mean_{ch}"]    = m
        feats[f"std_{ch}"]     = s
        feats[f"uni_{ch}"]     = float(s / (abs(m) + 1e-6))
        feats[f"entropy_{ch}"] = float(shannon_entropy(px))
        px_fin = px[np.isfinite(px)]
        feats[f"skew_{ch}"]    = (
            float(scipy.stats.skew(px_fin, bias=False))
            if px_fin.size > 2 else 0.0
        )
    return feats

N_JOBS = 16
LUCIA_ROOT  : /storage/homefs/db98d082/ondemand/LUCIA
H5_STACKS   : /storage/homefs/db98d082/ondemand/LUCIA/data/processed/cell_stacks.h5  exists=True
H5_MASKS    : /storage/homefs/db98d082/ondemand/LUCIA/data/processed/cell_masks.h5  exists=True
Tile count  : 162  (expect 162)
Channels    : ['el_lo', 'el_hi', 'pl_hi', 'pl_lo', 'rs_map', 'log_el_pl', 'grad_el_hi']
Schema      : 2 fixed + 5 stats x 7 channels = 37 cols


In [2]:
import os, numpy as np, h5py, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- 1. measure overlaps directly from FEATURE_TILES coordinates -------------
def _overlaps(tiles):
    xo, yo = [], []
    for half in sorted({t['half'] for t in tiles}):
        ts = [t for t in tiles if t['half'] == half]
        for r0 in sorted({t['row0'] for t in ts}):                      # horizontal neighbours
            row = sorted([t for t in ts if t['row0'] == r0], key=lambda t: t['col0'])
            xo += [a['col1'] - b['col0'] for a, b in zip(row, row[1:])]
        for c0 in sorted({t['col0'] for t in ts}):                      # vertical neighbours
            col = sorted([t for t in ts if t['col0'] == c0], key=lambda t: t['row0'])
            yo += [a['row1'] - b['row0'] for a, b in zip(col, col[1:])]
    return np.array(xo), np.array(yo)

_th = FEATURE_TILES[0]['row1'] - FEATURE_TILES[0]['row0']
_tw = FEATURE_TILES[0]['col1'] - FEATURE_TILES[0]['col0']
xo, yo = _overlaps(FEATURE_TILES)
print(f"TILING CHECK — {len(FEATURE_TILES)} tiles of {_th}x{_tw}px")
print(f"  horizontal overlap px: min={xo.min()} max={xo.max()} mean={xo.mean():.2f}")
print(f"  vertical   overlap px: min={yo.min()} max={yo.max()} mean={yo.mean():.2f}")
print("  EXPECT ~2 px uniform. If 0 → grid is non-overlapping (the old bug). "
      "If last-tile only is large → uneven fill.")

# ---- 2. load one real cell from the H5 --------------------------------------
geo   = pd.read_parquet(lc.GEOMETRY_PARQUET)
cn    = geo.index[geo['geom_keep']][0] if 'geom_keep' in geo.columns else geo.index[0]
row   = int(geo.loc[cn, 'npy_index'])
with h5py.File(lc.STACKS_H5, 'r') as f: el_hi = f['stacks'][row, 1].astype(np.float32)
with h5py.File(lc.MASKS_H5,  'r') as f: mask  = f['masks'][row].astype(np.float32)

# ---- 3. draw + save ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(el_hi, cmap='gray', aspect='equal')
ax.contour(mask > 127, levels=[0.5], colors='cyan', linewidths=1.2)
for t in FEATURE_TILES:
    excl = (t.get('active_frac', 1.0) < 0.5)
    ax.add_patch(Rectangle((t['col0'], t['row0']), t['col1']-t['col0'], t['row1']-t['row0'],
                 fill=excl, facecolor=('0.5' if excl else 'none'),
                 alpha=(0.35 if excl else 1.0),
                 edgecolor=('dimgray' if excl else 'yellow'), lw=0.6))
ax.set_title(f"NB1b tiling — {cn} | {len(FEATURE_TILES)} tiles {_th}px "
             f"overlap≈{xo.mean():.1f}px | active {el_hi.shape[0]}x{el_hi.shape[1]}", fontsize=10)
ax.set_xlabel("col (px)"); ax.set_ylabel("row (px)")

_fig_dir = (lc.FIGURES if hasattr(lc,'FIGURES') and os.path.isdir(getattr(lc,'FIGURES',''))
            else os.path.join(os.environ.get('LUCIA_ROOT','.'), 'outputs', 'figures'))
os.makedirs(_fig_dir, exist_ok=True)
_out = os.path.join(_fig_dir, 'nb1b_tiling_overlay.png')
fig.savefig(_out, dpi=200, bbox_inches='tight'); plt.close(fig)
print("Saved (open this file; Agg backend = no inline display):", _out)

TILING CHECK — 162 tiles of 64x64px
  horizontal overlap px: min=3 max=3 mean=3.00
  vertical   overlap px: min=3 max=4 mean=3.25
  EXPECT ~2 px uniform. If 0 → grid is non-overlapping (the old bug). If last-tile only is large → uneven fill.
Saved (open this file; Agg backend = no inline display): /storage/homefs/db98d082/ondemand/LUCIA/outputs/figures/nb1b_tiling_overlay.png


In [2]:
# ── Stage H5 files to node-local scratch ($TMPDIR) ───────────────────────────
# Network FS read is the dominant bottleneck on UBELIX.  Copy both H5 files
# once to local NVMe (TMPDIR), then all workers read from scratch.

_SCRATCH = os.environ.get('TMPDIR', os.path.join(lc.PROCESSED, '_scratch_ubx'))
os.makedirs(_SCRATCH, exist_ok=True)

H5_STACKS_LOCAL      = os.path.join(_SCRATCH, 'cell_stacks.h5')
H5_MASKS_LOCAL       = os.path.join(_SCRATCH, 'cell_masks.h5')
TILE_PARQUET_SCRATCH = os.path.join(_SCRATCH, 'lucia_tile_features.parquet')

for src, dst in [(H5_STACKS_SRC, H5_STACKS_LOCAL),
                 (H5_MASKS_SRC,  H5_MASKS_LOCAL)]:
    if os.path.exists(dst):
        sz = os.path.getsize(dst) / 1e9
        print(f"Already staged: {os.path.basename(dst)}  ({sz:.1f} GB)")
        continue
    sz_gb = os.path.getsize(src) / 1e9
    print(f"Staging {os.path.basename(src)}  ({sz_gb:.1f} GB) ...")
    t0 = time.time()
    shutil.copy2(src, dst)
    dt = time.time() - t0
    print(f"  Done in {dt:.0f}s  ({sz_gb/max(dt,0.1):.1f} GB/s)")

print(f"\nScratch dir : {_SCRATCH}")
print(f"  stacks  exists={os.path.exists(H5_STACKS_LOCAL)}")
print(f"  masks   exists={os.path.exists(H5_MASKS_LOCAL)}")

Already staged: cell_stacks.h5  (20.7 GB)
Already staged: cell_masks.h5  (0.1 GB)

Scratch dir : /scratch/local/7098716
  stacks  exists=True
  masks   exists=True


In [3]:
# ── Build work list + sanity gate (3-cell check) ─────────────────────────────
# Confirm npy_index maps cell_name → correct H5 row by comparing freshly
# computed tile stats against the existing TILE_PARQUET on PROCESSED.
# STOP and raise if mismatch — do not regenerate on a bad index map.

geo  = pd.read_parquet(GEOMETRY_PARQUET)
geo  = geo[geo['geom_keep']].copy()
work = list(geo['npy_index'].items())   # [(cell_name, h5_row), ...]

print(f"Geometry keep : {len(geo):,} cells")
print(f"Work list     : {len(work):,}")

_sanity_ok = False

if not os.path.exists(TILE_PARQUET):
    print("\nNo existing TILE_PARQUET — skipping comparison gate.")
    print("Verify npy_index mapping manually before the full run.")
    _sanity_ok = True
else:
    tf_old    = pd.read_parquet(TILE_PARQUET)
    _sample   = [cn for cn, _ in work
                 if cn in tf_old.index.get_level_values('cell_name')][:3]
    _TOL      = 0.05   # 5% relative tolerance
    _mismatches = []
    print(f"\nSanity gate: {len(_sample)} cells vs existing parquet ...")

    for cell_name, h5_row in work:
        if cell_name not in _sample:
            continue
        with h5py.File(H5_STACKS_LOCAL, 'r', swmr=True) as fst, \
             h5py.File(H5_MASKS_LOCAL,  'r', swmr=True) as fmk:
            stack4 = fst['stacks'][h5_row]
            mask   = fmk['masks'][h5_row]
        synth = lc.synthesize_channels(stack4)
        ch_arrays = {
            'el_lo':      stack4[0].astype(np.float32),
            'el_hi':      stack4[1].astype(np.float32),
            'pl_hi':      stack4[2].astype(np.float32),
            'pl_lo':      stack4[3].astype(np.float32),
            'rs_map':     synth[0],
            'log_el_pl':  synth[1],
            'grad_el_hi': synth[2],
        }
        cell_mask = mask > 127
        old_tiles = tf_old.loc[cell_name]
        for t in FEATURE_TILES:
            tid = t['tile_id']
            if tid not in old_tiles.index:
                continue
            r0, c0, r1, c1 = t['row0'], t['col0'], t['row1'], t['col1']
            tmask = cell_mask[r0:r1, c0:c1]
            if tmask.sum() == 0:
                continue
            tile_px = {ch: ch_arrays[ch][r0:r1, c0:c1][tmask] for ch in ALL_CHANNELS}
            feats   = _tile_features(tile_px, ALL_CHANNELS)
            old_row = old_tiles.loc[tid]
            for col in ('mean_el_hi', 'std_el_hi', 'entropy_el_hi'):
                if col not in old_row.index:
                    continue
                new_v = feats.get(col, float('nan'))
                old_v = float(old_row[col])
                if abs(new_v - old_v) / (abs(old_v) + 1e-6) > _TOL:
                    _mismatches.append((cell_name, tid, col, old_v, new_v))
            break  # one tile per cell suffices

    if _mismatches:
        print("SANITY GATE FAILED — npy_index mapping wrong or synthesize_channels changed:")
        for m in _mismatches:
            print(f"  {m[0]}  tile={m[1]}  {m[2]}:  old={m[3]:.4f}  new={m[4]:.4f}")
        raise RuntimeError("Sanity gate failed — do NOT run the batch.")
    else:
        print(f"SANITY GATE PASSED ({len(_sample)} cells, key stats within {_TOL*100:.0f}%)")
        _sanity_ok = True

assert _sanity_ok, "Sanity gate did not complete — check output above."

Geometry keep : 10,492 cells
Work list     : 10,492

No existing TILE_PARQUET — skipping comparison gate.
Verify npy_index mapping manually before the full run.


In [4]:
# ── Full regen prep — back up then delete old TILE_PARQUET ───────────────────
# Schema changed 22 → 37 cols.  Mixing old/new rows via concat produces ragged
# frames with NaN columns.  Geometry parquet and H5 stacks are unchanged.

assert _sanity_ok, "Run the sanity gate cell first."

if os.path.exists(TILE_PARQUET):
    bak = TILE_PARQUET + '.pre_regen_bak'
    shutil.copy2(TILE_PARQUET, bak)
    os.remove(TILE_PARQUET)
    print(f"Backed up  → {bak}")
    print(f"Deleted    : {TILE_PARQUET}")
else:
    print(f"Not found (clean state): {TILE_PARQUET}")

print("\nReady for full regen.  Run the batch cell next.")

Not found (clean state): /storage/homefs/db98d082/ondemand/LUCIA/data/processed/lucia_tile_features.parquet

Ready for full regen.  Run the batch cell next.


In [10]:
# ── Batch run + copy-back + schema verification ───────────────────────────────
# Each joblib worker opens H5 files independently.
# h5py.File is NOT fork-safe — never share an open file handle across workers.

def _worker(args):
    """Per-cell tile feature extraction from H5. Returns list of tile dicts."""
    cell_name, h5_row = args
    try:
        with h5py.File(H5_STACKS_LOCAL, 'r', swmr=True) as fst, \
             h5py.File(H5_MASKS_LOCAL,  'r', swmr=True) as fmk:
            stack4 = fst['stacks'][h5_row]   # (4, 558, 1108) uint8
            mask   = fmk['masks'][h5_row]    # (558, 1108) uint8
    except Exception:
        return []
    synth = lc.synthesize_channels(stack4)
    ch_arrays = {
        'el_lo':      stack4[0].astype(np.float32),
        'el_hi':      stack4[1].astype(np.float32),
        'pl_hi':      stack4[2].astype(np.float32),
        'pl_lo':      stack4[3].astype(np.float32),
        'rs_map':     synth[0],
        'log_el_pl':  synth[1],
        'grad_el_hi': synth[2],
    }
    cell_mask = mask > 127
    rows = []
    for t in FEATURE_TILES:
        r0, c0, r1, c1 = t['row0'], t['col0'], t['row1'], t['col1']
        tmask = cell_mask[r0:r1, c0:c1]
        if tmask.sum() == 0:
            continue
        tile_px = {ch: ch_arrays[ch][r0:r1, c0:c1][tmask] for ch in ALL_CHANNELS}
        feats   = _tile_features(tile_px, ALL_CHANNELS)
        rows.append(dict(
            cell_name=cell_name,
            **{k: t[k] for k in ('tile_id', 'row0', 'col0', 'row1', 'col1',
                                  'center_x', 'center_y', 'half',
                                  'is_border', 'active_frac')},
            **feats,
        ))
    return rows


print(f"Batch: {len(work):,} cells  N_JOBS={N_JOBS}  backend=loky")

t0 = time.time()
with joblib.Parallel(n_jobs=N_JOBS, backend='loky') as parallel:
    results = parallel(
        joblib.delayed(_worker)(args)
        for args in tqdm(work, mininterval=30, desc="tile extraction")
    )
dt = time.time() - t0

n_empty   = sum(1 for r in results if not r)
tile_rows = [row for cell_rows in results for row in cell_rows]
print(f"\nDone in {dt/60:.1f} min  |  cells with no output: {n_empty}")

if not tile_rows:
    raise RuntimeError("No tile rows produced — check H5 paths and worker logs above.")

tile_df = pd.DataFrame(tile_rows).set_index(['cell_name', 'tile_id'])

# ── Variance-floor guard: near-constant tiles make uni/skew/entropy unreliable ──
# (catastrophic-cancellation warnings).  Scrub non-finite + clip to 0 where unstable.
_CHANNELS = ['el_lo','el_hi','pl_hi','pl_lo','rs_map','log_el_pl','grad_el_hi']
_unstable = [f'{s}_{c}' for c in _CHANNELS for s in ('uni','skew','entropy')]
_unstable = [c for c in _unstable if c in tile_df.columns]
_n_bad = (~np.isfinite(tile_df[_unstable])).to_numpy().sum()
tile_df[_unstable] = tile_df[_unstable].replace([np.inf, -np.inf], np.nan).fillna(0.0)
print(f"Variance-floor scrub: set {_n_bad:,} non-finite uni/skew/entropy values → 0.0")

# ── Schema verification (44 total = 7 geom + 2 flags + 35 stats) ──────────────
_GEOM  = ['row0','col0','row1','col1','center_x','center_y','half']
_FLAGS = ['is_border','active_frac']
_STATS = ['mean','std','uni','entropy','skew']
_feat  = [f'{s}_{c}' for c in _CHANNELS for s in _STATS]          # 35
_expected = _GEOM + _FLAGS + _feat                                # 44
_missing  = [c for c in _expected if c not in tile_df.columns]
_extra    = [c for c in tile_df.columns if c not in _expected]
assert not _missing, f"Missing columns: {_missing}"
assert not _extra,   f"Unexpected columns: {_extra}"
assert tile_df.shape[1] == 44, f"Expected 44 total cols, got {tile_df.shape[1]}"
print(f"Schema: {tile_df.shape[1]} cols (7 geom + 2 flags + 35 stats)  OK")

_out_cells = set(tile_df.index.get_level_values('cell_name'))
_stray = _out_cells - set(geo.index)
assert not _stray, f"{len(_stray)} cell_names not in geometry: {list(_stray)[:5]}"
print(f"Row map: all {len(_out_cells):,} cell_names in geo.index  OK")

_tpc = tile_df.groupby('cell_name').size()
print(f"Tiles/cell: min={_tpc.min()}  max={_tpc.max()}  median={_tpc.median():.0f}  (expect ~162)")

# ── Save to scratch, then atomically move into PROCESSED ──────────────────────
tile_df.to_parquet(TILE_PARQUET_SCRATCH, engine='pyarrow')
print(f"Scratch parquet: {len(tile_df):,} rows  cols={tile_df.shape[1]}")

# back up any existing processed parquet, then move scratch → processed
if os.path.exists(TILE_PARQUET):
    _bak = TILE_PARQUET + '.22col_bak'
    if not os.path.exists(_bak):
        os.replace(TILE_PARQUET, _bak)
        print(f"Backed up old → {_bak}")
    else:
        os.remove(TILE_PARQUET)
try:
    os.replace(TILE_PARQUET_SCRATCH, TILE_PARQUET)   # atomic if same filesystem
except OSError:
    shutil.copy2(TILE_PARQUET_SCRATCH, TILE_PARQUET) # cross-FS fallback
    os.remove(TILE_PARQUET_SCRATCH)
sz_mb = os.path.getsize(TILE_PARQUET) / 1e6
print(f"Saved → {TILE_PARQUET}  ({sz_mb:.1f} MB)")
print("\nNB1b-UBX complete.")
print("Next: re-run NB3 (n_feat auto-sizes 22→44-col input, 35 feats + flags), then NB4, then NB7.")

Batch: 10,492 cells  N_JOBS=16  backend=loky


tile extraction:   0%|          | 0/10492 [00:00<?, ?it/s]/scratch/local/7098716/ipykernel_1283118/220028816.py:65: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
/scratch/local/7098716/ipykernel_1283118/220028816.py:65: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
/scratch/local/7098716/ipykernel_1283118/220028816.py:65: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
/scratch/local/7098716/ipykernel_1283118/220028816.py:65: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
/scratch/local/7098716/ipy


Done in 7.4 min  |  cells with no output: 0
Variance-floor scrub: set 49,134 non-finite uni/skew/entropy values → 0.0
Schema: 44 cols (7 geom + 2 flags + 35 stats)  OK
Row map: all 10,492 cell_names in geo.index  OK
Tiles/cell: min=162  max=162  median=162  (expect ~162)
Scratch parquet: 1,699,704 rows  cols=44
Saved → /storage/homefs/db98d082/ondemand/LUCIA/data/processed/lucia_tile_features.parquet  (414.2 MB)

NB1b-UBX complete.
Next: re-run NB3 (n_feat auto-sizes 22→44-col input, 35 feats + flags), then NB4, then NB7.


In [6]:
print(list(tile_df.columns))

['row0', 'col0', 'row1', 'col1', 'center_x', 'center_y', 'half', 'is_border', 'active_frac', 'mean_el_lo', 'std_el_lo', 'uni_el_lo', 'entropy_el_lo', 'skew_el_lo', 'mean_el_hi', 'std_el_hi', 'uni_el_hi', 'entropy_el_hi', 'skew_el_hi', 'mean_pl_hi', 'std_pl_hi', 'uni_pl_hi', 'entropy_pl_hi', 'skew_pl_hi', 'mean_pl_lo', 'std_pl_lo', 'uni_pl_lo', 'entropy_pl_lo', 'skew_pl_lo', 'mean_rs_map', 'std_rs_map', 'uni_rs_map', 'entropy_rs_map', 'skew_rs_map', 'mean_log_el_pl', 'std_log_el_pl', 'uni_log_el_pl', 'entropy_log_el_pl', 'skew_log_el_pl', 'mean_grad_el_hi', 'std_grad_el_hi', 'uni_grad_el_hi', 'entropy_grad_el_hi', 'skew_grad_el_hi']
